# 00 - Setup: Carga de Dados no Postgres

Este notebook carrega os arquivos CSV da pasta `data/` para o banco de dados **sistema_ouvidoria** no SQL Server 2025.

**Pré-requisitos:**
- Docker Compose rodando (`docker compose up -d`)
- Postgres acessível na porta `5432`

## 1. Configuração e Conexão

In [1]:
# Instalação da lib ODBC (para conexão com bancos relacionais) e Driver ODBC para Postgres
!apt-get update && apt-get install -y unixodbc unixodbc-dev odbc-postgresql

import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv

load_dotenv(override=True)

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

print(f'Servidor: {DB_SERVER}:{DB_PORT}')
print(f'Database: {DB_DATABASE}')

Get:1 http://deb.debian.org/debian trixie InRelease [140 kB]
Get:2 http://deb.debian.org/debian trixie-updates InRelease [47.3 kB]
Get:3 http://deb.debian.org/debian-security trixie-security InRelease [43.4 kB]
Get:4 http://deb.debian.org/debian trixie/main amd64 Packages [9671 kB]
Get:5 http://deb.debian.org/debian trixie-updates/main amd64 Packages [5412 B]
Get:6 http://deb.debian.org/debian-security trixie-security/main amd64 Packages [130 kB]
Fetched 10.0 MB in 1s (8415 kB/s)                         
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libltdl7 libodbc2 libodbccr2 libodbcinst2 libpq5 odbcinst unixodbc-common
Suggested packages:
  tdsodbc
The following NEW packages will be installed:
  libltdl7 libodbc2 libodbccr2 libodbcinst2 libpq5 odbc-postgresql odbcinst
  unixodbc unixodbc-common unixodbc-dev
0 upgraded, 10 newly installed, 0 to remov

In [2]:
# Conexão ao Postgres (master) para criar o database
conn_master = pyodbc.connect(
    f'DRIVER={{PostgreSQL Unicode}};'
    f'SERVER={DB_SERVER};'
    f'PORT={DB_PORT};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};',
    autocommit=True
)
cursor_master = conn_master.cursor()
print('Conectado ao PostgreSQL (master) com sucesso!')

Conectado ao PostgreSQL (master) com sucesso!


## 2. Criar Database sistema_ouvidoria

In [3]:
# Criar o database se não existir
cursor_master.execute(f"SELECT 1 FROM pg_database WHERE datname = '{DB_DATABASE}'")
databaseAlreadyExists = cursor_master.fetchone() is not None
if (databaseAlreadyExists):
    print(f'Database [{DB_DATABASE}] verificado com sucesso!')
else:
    cursor_master.execute(f"CREATE DATABASE {DB_DATABASE}")
    print(f'Database [{DB_DATABASE}] criado com sucesso!')

cursor_master.close()
conn_master.close()

Database [sistema_ouvidoria] criado com sucesso!


In [4]:
# Conectar ao database sistema_ouvidoria
conn = pyodbc.connect(
    f'DRIVER={{PostgreSQL Unicode}};'
    f'SERVER={DB_SERVER};'
    f'PORT={DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};',
    autocommit=True
)
cursor = conn.cursor()
print(f'Conectado ao [{DB_DATABASE}] com sucesso!')

Conectado ao [sistema_ouvidoria] com sucesso!


## 3. Criar Tabelas

In [5]:
# DDL - Criação das tabelas
ddl_statements = [
    # Tabelas de domínio (sem FK)
    """
    CREATE TABLE IF NOT EXISTS estado (
        id_estado SERIAL PRIMARY KEY,
        nome_estado VARCHAR(100) NOT NULL,
        sigla_estado CHAR(2) NOT NULL
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS cidade (
        id_cidade SERIAL PRIMARY KEY,
        nome_cidade VARCHAR(255) NOT NULL,
        cod_estado INT NOT NULL,
        CONSTRAINT fk_cidade_estado FOREIGN KEY (cod_estado) REFERENCES estado(id_estado)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS usuario (
        id_usuario SERIAL PRIMARY KEY,
        nome_usuario VARCHAR(255) NOT NULL,
        cpf_usuario VARCHAR(11) NOT NULL,
        email_usuario VARCHAR(255) NOT NULL,
        senha_usuario VARCHAR(255) NOT NULL,
        telefone_usuario VARCHAR(16),
        whatsapp_usuario VARCHAR(16),
        data_nasc DATE,
        cod_cidade INT NOT NULL,
        data_ultimo_acesso TIMESTAMP,
        hash_ativacao_usuario VARCHAR(255),
        CONSTRAINT fk_usuario_cidade FOREIGN KEY (cod_cidade) REFERENCES cidade(id_cidade)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS tipo_ouvidoria (
        id_tipo SERIAL PRIMARY KEY,
        nome_tipo VARCHAR(100) NOT NULL,
        descricao_tipo TEXT NOT NULL
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS servico_afetado (
        id_servico SERIAL PRIMARY KEY,
        nome_servico VARCHAR(255) NOT NULL
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS ouvidoria (
        id_ouvidoria SERIAL PRIMARY KEY,
        descricao_ouvidoria TEXT NOT NULL,
        cod_tipo INT NOT NULL,
        cod_servico INT NOT NULL,
        protocolo_ouvidoria VARCHAR(50) NOT NULL,
        data_ouvidoria TIMESTAMP NOT NULL,
        cod_usuario INT NOT NULL,
        CONSTRAINT fk_ouvidoria_tipo FOREIGN KEY (cod_tipo) REFERENCES tipo_ouvidoria(id_tipo),
        CONSTRAINT fk_ouvidoria_servico FOREIGN KEY (cod_servico) REFERENCES servico_afetado(id_servico),
        CONSTRAINT fk_ouvidoria_usuario FOREIGN KEY (cod_usuario) REFERENCES usuario(id_usuario)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS anexo (
        id_anexo SERIAL PRIMARY KEY,
        nome_anexo VARCHAR(255) NOT NULL,
        arquivo_anexo TEXT NOT NULL,
        cod_ouvidoria INT NOT NULL,
        CONSTRAINT fk_anexo_ouvidoria FOREIGN KEY (cod_ouvidoria) REFERENCES ouvidoria(id_ouvidoria)
    )
    """
]

for ddl in ddl_statements:
    cursor.execute(ddl)
    
print('Todas as tabelas foram criadas com sucesso!')

Todas as tabelas foram criadas com sucesso!


## 4. Carregar Dados dos CSVs

In [6]:
# Ordem de carga (respeitar dependências)
tabelas = ['estado', 'cidade', 'usuario', 'tipo_ouvidoria', 'servico_afetado', 'ouvidoria', 'anexo']

data_dir = os.path.join(os.getcwd(), 'data')

for tabela in tabelas:
    csv_path = os.path.join(data_dir, f'{tabela}.csv')
    
    # Verificar se a tabela já tem dados
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    
    if count > 0:
        print(f'  ⏭️  {tabela}: já contém {count} registros, pulando...')
        continue
    
    # Ler CSV
    df = pd.read_csv(csv_path)
    
    # Limpar espaços em colunas string
    for col in df.select_dtypes(include=['str', 'object']).columns:
        df[col] = df[col].str.strip()
    
    # Inserir dados via executemany
    cols = ', '.join(df.columns)
    placeholders = ', '.join(['?' for _ in df.columns])
    insert_sql = f'INSERT INTO {tabela} ({cols}) VALUES ({placeholders})'
    
    # Converter NaN para None
    data = [tuple(None if pd.isna(v) else v for v in row) for row in df.itertuples(index=False)]
    
    # Inserir em lotes de 1000
    batch_size = 2000
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        cursor.executemany(insert_sql, batch)
    
    print(f'  ✅ {tabela}: {len(data)} registros inseridos')

print('\n🎉 Carga de dados concluída!')

  ✅ estado: 27 registros inseridos
  ✅ cidade: 500 registros inseridos
  ✅ usuario: 3 registros inseridos
  ✅ tipo_ouvidoria: 5 registros inseridos
  ✅ servico_afetado: 8 registros inseridos
  ✅ ouvidoria: 3 registros inseridos
  ✅ anexo: 2 registros inseridos

🎉 Carga de dados concluída!


## 5. Validação

In [7]:
# Verificar contagem de registros em cada tabela
print(f'{"Tabela":<20} {"Registros":>10}')
print('-' * 32)

for tabela in tabelas:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    print(f'{tabela:<20} {count:>10}')

print('\n✅ Validação concluída!')

Tabela                Registros
--------------------------------
estado                       27
cidade                      500
usuario                       3
tipo_ouvidoria                5
servico_afetado               8
ouvidoria                     3
anexo                         2

✅ Validação concluída!


In [8]:
# Amostra de dados de algumas tabelas
for tabela in ['cidade', 'servico_afetado', 'ouvidoria']:
    print(f'\n--- {tabela.upper()} (primeiros 5 registros) ---')
    df_sample = pd.read_sql(f'SELECT * FROM {tabela} LIMIT 5', conn)
    print(df_sample.to_string(index=False))


--- CIDADE (primeiros 5 registros) ---
 id_cidade        nome_cidade  cod_estado
         1     Afonso Cláudio           8
         2 Água Doce do Norte           8
         3       Águia Branca           8
         4             Alegre           8
         5     Alfredo Chaves           8

--- SERVICO_AFETADO (primeiros 5 registros) ---
 id_servico                   nome_servico
          1 Educação, Ciência e Tecnologia
          2                          Saúde
          3                   Procuradoria
          4                        Fazenda
          5                    Agricultura

--- OUVIDORIA (primeiros 5 registros) ---
 id_ouvidoria                      descricao_ouvidoria  cod_tipo  cod_servico protocolo_ouvidoria      data_ouvidoria  cod_usuario
            1          Falta de medicamentos no posto.         4            2        202605010001 2026-05-01 08:30:00            1
            2       Ótimo atendimento dos professores.         2            1        20260501000

/tmp/ipykernel_9/2004273020.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT * FROM {tabela} LIMIT 5', conn)
/tmp/ipykernel_9/2004273020.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT * FROM {tabela} LIMIT 5', conn)
/tmp/ipykernel_9/2004273020.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT * FROM {tabela} LIMIT 5', conn)


In [9]:
# Encerrar conexão
cursor.close()
conn.close()
print('Conexão encerrada.')

Conexão encerrada.
